In [ ]:
import os
import json
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

MODEL = "openai/gpt-oss-120b"
QUERY = "How does the agentic loop keep calling the model until it stops?"

client = Groq()

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
files = reader.read()

documents = []
for file in files:
    documents.append(file.parse())


print("Q1 - number of lesson pages:", len(documents))



index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

q2_results = index.search(QUERY, num_results=5)
print("Q2 - filename of first result:", q2_results[0]["filename"])


def build_context(results):
    parts = []
    for doc in results:
        parts.append(f"filename: {doc['filename']}\n{doc['content']}")
    return "\n\n---\n\n".join(parts)


def rag(query, search_index, num_results=5):
    """Returns (answer_text, input_tokens)."""
    results = search_index.search(query, num_results=num_results)
    context = build_context(results)

    prompt = (
        "You're a course teaching assistant. Answer the QUESTION using only "
        "the CONTEXT from the course lessons.\n\n"
        f"QUESTION: {query}\n\n"
        f"CONTEXT:\n{context}"
    )

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    answer = response.choices[0].message.content
    input_tokens = response.usage.prompt_tokens
    return answer, input_tokens



q3_answer, q3_tokens = rag(QUERY, index)
print("Q3 - input (prompt) tokens:", q3_tokens)


chunks = chunk_documents(documents, size=2000, step=1000)
print("Q4 - number of chunks:", len(chunks))



chunk_index = Index(text_fields=["content"], keyword_fields=["filename"])
chunk_index.fit(chunks)

q5_answer, q5_tokens = rag(QUERY, chunk_index)
print("Q5 - input tokens (chunked):", q5_tokens)
print("Q5 - fewer tokens than Q3:", q3_tokens - q5_tokens,
      f"(~{q3_tokens / max(q5_tokens, 1):.1f}x fewer)")


AGENT_QUERY = "How does the agentic loop work, and how is it different from plain RAG?"
AGENT_INSTRUCTIONS = (
    "You're a course teaching assistant. Answer the student's question using "
    "the search tool. Make multiple searches with different keywords before "
    "answering."
)


def search(query: str) -> str:
    """Search the course lessons for relevant chunks given a query string."""
    results = chunk_index.search(query, num_results=5)
    return build_context(results)


tools = [{
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the course lessons for relevant content.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search keywords"}
            },
            "required": ["query"],
        },
    },
}]

messages = [
    {"role": "system", "content": AGENT_INSTRUCTIONS},
    {"role": "user", "content": AGENT_QUERY},
]

search_call_count = 0
for _ in range(20):
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
    )
    msg = response.choices[0].message

    messages.append({
        "role": "assistant",
        "content": msg.content or "",
        "tool_calls": [
            {
                "id": tc.id,
                "type": "function",
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                },
            }
            for tc in (msg.tool_calls or [])
        ] or None,
    })

    if not msg.tool_calls:
        break

    for tool_call in msg.tool_calls:
        if tool_call.function.name == "search":
            search_call_count += 1
            args = json.loads(tool_call.function.arguments)
            result = search(args["query"])
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            })

print("Q6 - number of search calls:", search_call_count)